# Generic Selector vs. Brute Force

Choose a 4-bit format by checking selected features, then compare that choice with a search using the full output.


## Main idea

A weight column pairs with an input row during multiplication. We select important features from both matrices and combine their indices. Each format is scored using these selected features. Brute force checks the complete output while trying different scales.


## Configuration

Set `X_CAL_PATH` to the calibration input and `WEIGHT_DATASETS` to the weight-file paths. Dataset names are display labels. All listed weights use the same calibration input.

Inputs must be finite, nonempty 2-D arrays with `W.shape[1] == X_cal.shape[0]`. Results are saved to `/kaggle/working/` on Kaggle, or the current directory elsewhere.


In [1]:
from pathlib import Path
import math, time, json, re
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

X_CAL_PATH = Path("c:/Users/Admin/OneDrive/Desktop/System/X_cal.npy")

WEIGHT_DATASETS = {
    "Dataset C": Path("c:/Users/Admin/OneDrive/Desktop/System/W_C.npy"),

}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

K_W_PCT = 20.0
K_X_PCT = 20.0
EXPORT_RESULTS = True

E8M0_MIN_EXP = - 127
E8M0_MAX_EXP = 127
BFP_EXP_POLICY = "autoflex_predict"
BFP_ALPHA = 1.0
BFP_SHARING = "block"
BFP_BLOCK_SIZE = 32
BFP_METADATA_BITS = 8

def show(df):

    with pd.option_context('display.max_rows', 20, 'display.max_columns', 20, 'display.precision', 6):
        display(df)


## Prepare the number formats

The next cell defines five ways to round weights: E2M1, E1M2, MXINT4, MXUINT4, and BFP4. Each format allows a small set of values. Groups of weights share a scale, which controls the step size. The cell also checks the inputs and provides the rounding functions.


In [2]:
import itertools
import json
import math
import time
from pathlib import Path
import numpy as np
import pandas as pd

def make_formats(bfp_block_size = 32, bfp_metadata_bits = 8):
    """Build the five 4-bit format configurations."""
    common = dict(
        element_bits = 4,
        metadata_bits = 8,
        block_size = 32,
        exp_min = E8M0_MIN_EXP,
        exp_max = E8M0_MAX_EXP,
    )
    return {
        "E2M1": dict(
            common, kind = "mxfp", layout = "[S|E|M] = [1|2|1]",
            grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], dtype = np.float64),
            baseline_exp_shift =  - 2,
            shared_scale = "power-of-two 2^e; 8-bit E8M0-style shared exponent",
            exceptions = "finite values only; no NaN/Inf codes are emitted",
            convention_status = "OCP MX E2M1 element codebook/convention used in this project",
        ),
        "E1M2": dict(
            common, kind = "mxfp", layout = "[S|E|M] = [1|1|2]",
            grid = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5], dtype = np.float64),
            baseline_exp_shift =  - 1,
            shared_scale = "power-of-two 2^e; 8-bit E8M0-style shared exponent",
            exceptions = "finite values only; no NaN/Inf codes are emitted",
            convention_status = "adopted classroom-consistent E1M2 codebook/scaling convention; not a concrete OCP MX v1.0 format",
        ),
        "MXINT4": dict(
            common, kind = "mxint", layout = "[S|I|F] = [1|1|2]",
            qmin =  - 8, qmax = 7, element_scale = 0.25,
            shared_scale = "power-of-two 2^e; element precision scale 2^-2",
            exceptions = "signed integer codes -8..7; no NaN/Inf codes",
            convention_status = "course/project MXINT4 construction; OCP MX v1.0's concrete integer format is MXINT8",
        ),
        "BFP4": dict(
            kind = "bfp", layout = "Flexpoint flex4+8: signed 4-bit two's-complement mantissa",
            element_bits = 4, metadata_bits = int(bfp_metadata_bits),
            block_size = int(bfp_block_size), mantissa_bits = 4, qmin =  - 8, qmax = 7,
            exp_min = E8M0_MIN_EXP, exp_max = E8M0_MAX_EXP,
            shared_scale = "kappa = 2^E; value = m * 2^E; 8-bit shared exponent per block",
            exceptions = "finite signed two's-complement mantissas; saturate to [-8,7]; no NaN/Inf codes",
            convention_status = "Flexpoint flex4+8 with Autoflex; assignment default uses row-wise block sharing",
        ),
        "MXUINT4": dict(
            common, kind = "mxuint", layout = "Unsigned 4-bit code; fixed midpoint decode q_s = q_u - 8",
            qmin = 0, qmax = 15, zero_point = 8, encoding_defined = True,
            shared_scale = "power-of-two 2^e; fixed zero point 8; no per-block FP32 offset",
            exceptions = "finite values; unsigned codes 0..15 decode to signed integers -8..7",
            convention_status = "custom midpoint-coded unsigned-storage convention; not a standardized OCP MXUINT4 format",
        ),
    }

def validate_formats(formats):

    for name, cfg in formats.items():
        if cfg["block_size"] < 1 or cfg["element_bits"] != 4:
            raise ValueError(f"{name}: need a positive block size and 4 element bits.")

        if cfg["metadata_bits"] < 1 or cfg["exp_min"] > cfg["exp_max"]:
            raise ValueError(f"{name}: invalid exponent/metadata settings.")

        if cfg["exp_max"] - cfg["exp_min"] + 1 > 2 ** cfg["metadata_bits"]:
            raise ValueError(f"{name}: exponent range does not fit stored metadata width.")

        if cfg["kind"] == "mxfp":
            grid = np.asarray(cfg["grid"], dtype = np.float64)
            if grid.ndim != 1 or len(grid) != 8 or grid[0] != 0 or np.any(np.diff(grid) <= 0):
                raise ValueError(f"{name}: expected eight increasing magnitudes starting at zero.")
        elif cfg["kind"] == "mxuint":
            required = {"qmin", "qmax", "zero_point", "encoding_defined"}
            if not required.issubset(cfg) or cfg["qmin"] != 0 or cfg["qmax"] != 15 or cfg["zero_point"] != 8:
                raise ValueError("MXUINT4 must use unsigned codes 0..15 with fixed midpoint zero point 8.")

def validated_arrays(W_input, X_input):

    for name, a in [("W", W_input), ("X_cal", X_input)]:
        if not isinstance(a, np.ndarray) or a.ndim != 2 or min(a.shape) == 0:
            raise ValueError(f"{name} must be a nonempty 2-D numeric array.")

        if not np.issubdtype(a.dtype, np.number) or np.iscomplexobj(a):
            raise ValueError(f"{name} must contain real numeric values.")

        if not np.isfinite(a).all():
            raise ValueError(f"{name} contains NaN/Inf.")

    if W_input.shape[1] != X_input.shape[0]:
        raise ValueError(f"Need W.shape[1] == X_cal.shape[0], got {W_input.shape}, {X_input.shape}.")

    return np.asarray(W_input, dtype = np.float64), np.asarray(X_input, dtype = np.float64)

def block_ranges(ncols, block_size):
    return [(s, min(s + block_size, ncols)) for s in range(0, ncols, block_size)]

def storage_for(cfg, shape):
    n = math.prod(shape)
    blocks = shape[0] * math.ceil(shape[1] / cfg["block_size"])
    total = n * cfg["element_bits"] + blocks * cfg["metadata_bits"]
    return dict(Blocks = blocks, Total_Storage_Bits = total, Bits_per_Weight = total / n)

def eligible_formats(W, formats, max_bits = None):
    rows, eligible = [], []

    for name, cfg in formats.items():
        st = storage_for(cfg, W.shape)
        status, reason = "ELIGIBLE", "Encoding is defined in the configuration."
        if cfg["kind"] == "mxuint" and not cfg.get("encoding_defined", False):
            status, reason = "NOT_SCORED", "Unsigned midpoint encoding is not defined."

        if status == "ELIGIBLE" and max_bits is not None and st["Bits_per_Weight"] > max_bits + 1e-12:
            status, reason = "OVER_STORAGE_LIMIT", "Effective storage exceeds the supplied limit."

        if status == "ELIGIBLE":
            eligible.append(name)

        rows.append(dict(Format = name, Status = status, Reason = reason, ** st))

    return eligible, pd.DataFrame(rows)

def nearest_grid_rne(values, grid):
    """Nearest grid value; ties choose an even grid index (as in brute force)."""
    values = np.asarray(values, dtype = np.float64)
    flat = values.ravel()
    out = np.empty_like(flat)

    for i, x in enumerate(flat):
        distance = np.abs(grid - x)
        dmin = np.min(distance)
        candidates = np.where(np.isclose(distance, dmin, rtol = 0.0, atol = 1e-15))[0]
        if len(candidates) == 1:
            idx = int(candidates[0])
        else:
            even = candidates[candidates % 2 == 0]
            idx = int(even[0] if len(even) else candidates[0])

        out[i] = grid[idx]

    return out.reshape(values.shape)

def _bfp_gamma(block, exponent):
    """Autoflex Gamma, matching the brute-force notebook's signed-mantissa rule."""
    nbits = FORMATS["BFP4"]["mantissa_bits"]
    mantissa = np.rint(np.abs(np.asarray(block, dtype = np.float64)) / np.exp2(float(exponent)))
    return float(min(np.max(mantissa), 2 ** (nbits - 1) - 1))

def bfp_exp_autoflex_predict(block):
    """Autoflex Algorithm 2 for a static block: E=ceil(log2(chi))-N+1."""
    nbits = FORMATS["BFP4"]["mantissa_bits"]
    chi = BFP_ALPHA * float(np.max(np.abs(block)))
    if chi == 0:
        return 0

    return int(np.ceil(np.log2(chi))) - nbits + 1

def bfp_exp_autoflex_init(block):
    """Autoflex trial-and-error initialization as implemented in bruteforce(3)."""
    nbits = FORMATS["BFP4"]["mantissa_bits"]
    if float(np.max(np.abs(block))) == 0:
        return 0

    exponent = 0

    for _ in range(1024):
        gamma = _bfp_gamma(block, exponent)
        if gamma >= 2 ** (nbits - 1) - 1:
            exponent += (nbits - 1) // 2
        elif gamma < 2 ** (nbits - 2):
            exponent += int(np.ceil(np.log2(max(gamma, 1.0)))) - (nbits - 2)
            if gamma > 2.0 ** ((nbits - 1) // 2 - 2):
                break
        else:
            break

        exponent = int(np.clip(exponent, E8M0_MIN_EXP, E8M0_MAX_EXP))

    return exponent

def bfp_shared_exponent(block):
    if BFP_EXP_POLICY == "autoflex_init":
        return bfp_exp_autoflex_init(block)

    if BFP_EXP_POLICY == "autoflex_predict":
        return bfp_exp_autoflex_predict(block)

    raise ValueError(f"Unknown BFP_EXP_POLICY={BFP_EXP_POLICY!r}")

def baseline_exponent(block, cfg):
    """Choose the baseline shared exponent for a block."""
    block = np.asarray(block, dtype = np.float64)
    amax = float(np.max(np.abs(block)))
    if amax == 0:
        return 0

    raw = int(np.floor(np.log2(amax)))

    if cfg["kind"] == "mxfp":
        exponent = raw + cfg["baseline_exp_shift"]
    elif cfg["kind"] == "mxuint":

        neg_extent = max(0.0, - float(np.min(block)))
        pos_extent = max(0.0, float(np.max(block)))
        required_step = max(neg_extent / 8.0, pos_extent / 7.0)
        exponent = int(np.ceil(np.log2(required_step))) if required_step > 0 else 0
    elif cfg["kind"] == "bfp":
        exponent = bfp_shared_exponent(block)
    else:
        exponent = raw

    return int(np.clip(exponent, cfg["exp_min"], cfg["exp_max"]))

def quantize_values(values, cfg, exponent):
    """Reconstruct quantized values using the same codebooks/RNE/saturation as brute force."""
    a = np.asarray(values, dtype = np.float64)
    exponent = np.asarray(np.clip(exponent, cfg["exp_min"], cfg["exp_max"]), dtype = np.int16)
    scale = np.exp2(exponent)

    if cfg["kind"] == "mxfp":
        local = np.abs(a) / scale
        qlocal = nearest_grid_rne(local, cfg["grid"])
        return np.sign(a) * qlocal * scale

    if cfg["kind"] == "mxint":
        step = scale * cfg["element_scale"]
        q_raw = np.rint(a / step)
        q = np.clip(q_raw, cfg["qmin"], cfg["qmax"])
        return q * step

    if cfg["kind"] == "bfp":

        m_raw = np.rint(a / scale)
        m = np.clip(m_raw, cfg["qmin"], cfg["qmax"])
        return m * scale

    if cfg["kind"] == "mxuint":

        q_raw = np.rint(a / scale) + cfg["zero_point"]
        q_u = np.clip(q_raw, cfg["qmin"], cfg["qmax"])
        q_signed = q_u - cfg["zero_point"]
        return q_signed * scale

    raise ValueError(f"Unknown quantizer kind: {cfg['kind']}")

def materialize(W, cfg, exponents):
    ranges = block_ranges(W.shape[1], cfg["block_size"])
    if exponents.shape != (W.shape[0], len(ranges)):
        raise ValueError("Exponent array and block layout disagree.")

    Wq = np.empty_like(W, dtype = np.float64)

    for r in range(W.shape[0]):

        for b, (s, t) in enumerate(ranges):
            Wq[r, s:t] = quantize_values(W[r, s:t], cfg, exponents[r, b])

    return Wq

def baseline_exponents(W, cfg):
    ranges = block_ranges(W.shape[1], cfg["block_size"])
    return np.array(
        [[baseline_exponent(W[r, s:t], cfg) for s, t in ranges]
         for r in range(W.shape[0])],
        dtype = np.int16,
    )

def relative_error_pct(sse, energy):
    if energy == 0:
        return 0. if sse == 0 else float("inf")

    return 100. * math.sqrt(max(0., sse) / energy)

if BFP_SHARING != "block":
    raise ValueError(
        "wandxtopk implements row-wise BFP block sharing, matching the brute-force notebook's default BFP_SHARING='block'."
    )

FORMATS = make_formats(BFP_BLOCK_SIZE, BFP_METADATA_BITS)
validate_formats(FORMATS)
FORMAT_CONFIGURATION = pd.DataFrame([
    dict(Format = f, Kind = c['kind'], Block_size = c['block_size'],
         Element_bits = c['element_bits'], Shared_metadata_bits = c['metadata_bits'],
         Exponent_min = c['exp_min'], Exponent_max = c['exp_max'],
         Baseline_exp_shift = c.get('baseline_exp_shift', 'format-specific rule'),
         Shared_scale = c.get('shared_scale', ''),
         Convention_status = c.get('convention_status', ''),
         Exceptions = c.get('exceptions', ''))
    for f, c in FORMATS.items()
])


## Load the inputs

Read the configured `.npy` files and check their shapes and values. No special filenames are required.


In [3]:
def load_npy_from_explicit_path(path, role):
    """Load exactly the file specified by the user; do not infer filenames."""
    p = Path(path).expanduser()
    if not p.is_absolute():
        p = Path.cwd() / p

    p = p.resolve()
    if not p.is_file():
        raise FileNotFoundError(
            f"{role} file was not found at:\n  {p}\n"
            "Edit X_CAL_PATH or WEIGHT_DATASETS in the configuration cell."
        )

    arr = np.load(p, allow_pickle = False)
    if not isinstance(arr, np.ndarray):
        raise ValueError(f"{role} file must contain one NumPy array: {p}")

    return arr, str(p)

if not isinstance(WEIGHT_DATASETS, dict) or not WEIGHT_DATASETS:
    raise ValueError("WEIGHT_DATASETS must be a non-empty dictionary of labels to full weight-file paths.")

if any(not isinstance(label, str) or not label.strip() for label in WEIGHT_DATASETS):
    raise ValueError("Every WEIGHT_DATASETS key must be a non-empty string label.")

x_input, x_path = load_npy_from_explicit_path(X_CAL_PATH, "Calibration input")
CASES = {}

for label, weight_path in WEIGHT_DATASETS.items():
    w_input, w_path = load_npy_from_explicit_path(weight_path, f"Weight matrix for {label!r}")
    W, X = validated_arrays(w_input, x_input)
    CASES[label] = dict(W = W, X = X, W_source = w_path, X_source = x_path)

if not CASES:
    raise ValueError("No weight datasets were loaded. Add at least one entry to WEIGHT_DATASETS.")

show(pd.DataFrame([
    dict(Dataset = label, W_shape = case['W'].shape,
         X_cal_shape = case['X'].shape,
         Weights = case['W_source'], Calibration = case['X_source'])
    for label, case in CASES.items()
]))


,Dataset,W_shape,X_cal_shape,Weights,Calibration
0,Dataset C,"(100, 100)","(100, 256)",C:\Users\Admin\OneDrive\Desktop\System\W_C.npy,C:\Users\Admin\OneDrive\Desktop\System\X_cal.npy


## Choose important features

A feature index connects one weight column with the matching input row in matrix multiplication. This method uses information from both sides to decide which features to keep.

1. Add the absolute values in each weight column. This gives one importance score per feature from `W`.
2. Add the absolute values in each input row. This gives a second importance score per feature from `X`.
3. Select the top `K_W_PCT` percent from the weight scores and the top `K_X_PCT` percent from the input scores. The number selected is rounded up when needed.
4. Combine the two index lists using a union. Overlapping indices appear only once. Keep the final features in their original order for calculation.

**Example:** weight-column scores are `[2, 9, 4]` and input-row scores are `[8, 3, 5]`. If each list selects one feature, the weights select feature 2 and the inputs select feature 1. The combined set is `{1, 2}`. If both selected feature 2, the combined set would contain only feature 2.

Absolute values prevent cancellation: `−4` and `3` contribute `4 + 3 = 7`, rather than `−1`. The actual union size determines how many weights are scored; it can be smaller than the sum of the two selection sizes.


In [4]:
def choose_columns(W, X, w_pct, x_pct):

    for pct in (w_pct, x_pct):
        if not np.isfinite(pct) or not 0 <= pct <= 100:
            raise ValueError('Percentages must be between 0 and 100.')

    features = W.shape[1]
    w_score = np.sum(np.abs(W), axis = 0)
    x_score = np.sum(np.abs(X), axis = 1)
    w_order = np.argsort( - w_score, kind = 'stable')
    x_order = np.argsort( - x_score, kind = 'stable')
    nw = int(math.ceil(features * w_pct / 100))
    nx = int(math.ceil(features * x_pct / 100))
    w_columns = w_order[:nw]
    x_rows = x_order[:nx]
    selected = np.union1d(w_columns, x_rows)
    if len(selected) == 0: raise ValueError('No features selected. Increase a percentage.')

    overlap = len(np.intersect1d(w_columns, x_rows))
    wrank = np.empty(features, int); wrank[w_order] = np.arange(1, features + 1)
    xrank = np.empty(features, int); xrank[x_order] = np.arange(1, features + 1)
    table = pd.DataFrame(dict(Original_feature = np.arange(features),
        W_absolute_sum = w_score, X_absolute_sum = x_score,
        W_rank = wrank, X_rank = xrank,
        Selected_from_W = np.isin(np.arange(features), w_columns),
        Selected_from_X = np.isin(np.arange(features), x_rows),
        Required_W_column = np.isin(np.arange(features), selected)))
    stats = dict(Selected_W_columns = nw, Selected_X_rows = nx,
        Before_overlap = nw + nx, Overlap_removed = overlap,
        Unique_W_columns = len(selected), Total_columns = features)
    return selected, table, stats


## Quantize the selected weights

The selected features can belong to several original weight blocks. For each block that contains a selected feature:

1. Read the complete original block and calculate its baseline shared scale.
2. Round only the selected weights using that scale and the current format.
3. Store the rounded values in a smaller selected-column matrix.

Using the complete original block keeps the scale consistent with full baseline quantization. We do not create new blocks by joining the selected columns together.

The returned trace records the scale and work counts. Values read to choose scales and values actually quantized are different counts.


In [5]:
def quantize_selected_columns(W, cfg, columns):
    sampled = np.empty((W.shape[0], len(columns)), dtype = np.float64)
    records = []
    count = 0

    for r in range(W.shape[0]):

        for b, (start, end) in enumerate(block_ranges(W.shape[1], cfg['block_size'])):
            positions = np.flatnonzero((columns >= start)&(columns < end))
            if len(positions) == 0: continue

            original = W[r, start:end]
            exponent = baseline_exponent(original, cfg)
            original_ids = columns[positions]
            values = W[r, original_ids]
            sampled[r, positions] = quantize_values(values, cfg, exponent)
            count += len(values)
            records.append(dict(Row = r, Block = b, Start = start, End = end,
                Original_max_abs = float(np.max(np.abs(original))),
                Shared_exponent = exponent, Weights_read_for_scale = end - start,
                Weights_quantized = len(values)))

    assert count == W.shape[0] * len(columns)
    return sampled, count, pd.DataFrame(records)


## Score formats and choose a winner

Each format is tested on the same selected features and calibration inputs.

First, subtract the rounded selected weights from the original selected weights. Multiply these weight errors by the matching input rows. This tells us how quantization changes the output contribution from the selected features.

Add the signed contributions before squaring. Then square the output errors and add them to obtain the score. Contributions can cancel, so squaring each individual contribution before adding would measure a different quantity.

The format with the smallest score wins. Exact score ties are resolved using storage, then format order. A separate indicator marks scores that are nearly equal within a small tolerance.

The reported error percentage is measured relative to the selected-feature output. It is a partial score, not the complete output error; omitted features can change the final ranking.


In [6]:
def select_format(W, X):
    begin = time.perf_counter()
    columns, importance, selection_stats = choose_columns(W, X, K_W_PCT, K_X_PCT)
    selected_W = W[:, columns]
    selected_X = X[columns, :]
    target = selected_W @ selected_X
    target_energy = float(np.sum(target ** 2))
    eligible, eligibility = eligible_formats(W, FORMATS)
    if not eligible: raise ValueError('No eligible formats.')

    scores = []; row_records = []; trace_records = []

    for fmt in eligible:
        cfg = FORMATS[fmt]
        t = time.perf_counter()
        Wq, count, trace = quantize_selected_columns(W, cfg, columns)
        quant_seconds = time.perf_counter() - t
        t = time.perf_counter()
        error = (selected_W - Wq) @ selected_X
        row_loss = np.sum(error ** 2, axis = 1)
        score_seconds = time.perf_counter() - t
        sse = float(row_loss.sum())
        scores.append(dict(Format = fmt, Sampled_SSE = sse, Sampled_error_pct = relative_error_pct(sse, target_energy),
            Weights_quantized = count, Full_weights = W.size,
            Scoring_terms = count * X.shape[1], Full_scoring_terms = W.size * X.shape[1],
            Reduction_pct = 100 * (1 - count / W.size),
            Scale_scan_values = int(trace.Weights_read_for_scale.sum()),
            Quantize_seconds = quant_seconds, Score_seconds = score_seconds,
            Effective_bits_per_weight = storage_for(cfg, W.shape)['Bits_per_Weight']))
        row_records.extend(dict(Format = fmt, Row = r, Sampled_row_SSE = float(v)) for r, v in enumerate(row_loss))
        trace.insert(0, 'Format', fmt); trace_records.append(trace)

    table = pd.DataFrame(scores).sort_values(['Sampled_SSE', 'Effective_bits_per_weight'], kind = 'stable').reset_index(drop = True)
    table.insert(0, 'Predicted_rank', np.arange(1, len(table) + 1))
    table['Tied_for_best'] = np.isclose(table.Sampled_SSE, table.iloc[0].Sampled_SSE, rtol = 1e-12, atol = 1e-12)
    return dict(columns = columns, importance = importance, selection_stats = selection_stats,
        scores = table, row_losses = pd.DataFrame(row_records), block_trace = pd.concat(trace_records, ignore_index = True),
        eligibility = eligibility, winner = table.iloc[0].Format, seconds = time.perf_counter() - begin)

RESULTS = {}

for label, case in CASES.items():
    RESULTS[label] = select_format(case['W'], case['X'])


## Search block scales and compare the results

The generic selector uses baseline scales. This reference search asks whether other block scales can reduce the complete output error.

For each format and weight row, build nearby exponent choices for each block, try their combinations, and keep the combination with the smallest error against `W @ X_cal`. Testing blocks together accounts for cancellation or reinforcement between their output errors.

The search starts within ±3 of the baseline exponents. If the best candidate reaches the boundary, it can widen up to ±5. It searches a bounded range, not every possible scale. The comparison raises an error if a row uses a heuristic fallback rather than exact enumeration.

The final table compares the generic choice with this reference and reports their errors and costs. The generic selected-feature percentage and the reference full-output percentage use different reference outputs, so their values should not be treated as the same measurement.


In [7]:
REF_FORMATS = ["E2M1", "E1M2", "MXUINT4", "MXINT4", "BFP4"]
REF_WINDOW = 3
REF_WINDOW_MAX = 5
REF_MAX_ELEMS = 2_000_000
REF_RANDOM_STARTS = 4
REF_PAIR_MOVES = True
REF_USE_GRAM_FACTOR = True
REF_CAL_SPLIT_FRAC = 0.25
REF_VALIDATE_HEURISTIC = True
REF_SEED = 0

FORMAT_CONFIGS = FORMATS
rowwise_ranges = block_ranges

def baseline_exp(block, fmt):
    return baseline_exponent(block, FORMAT_CONFIGS[fmt])

def quantize_block_at_exp(block, fmt, exp, return_events = False):

    return quantize_values(block, FORMAT_CONFIGS[fmt], exp)

def _gram_factor(X):
    """Return F such that F F^T = X X^T on the non-zero eigenspace."""
    G = X @ X.T
    lam, V = np.linalg.eigh(G)

    if lam.size == 0 or np.max(lam) <= 0:
        return np.zeros((X.shape[0], 0), dtype = np.float64)

    keep = lam > np.max(lam) * 1e-14
    return V[:, keep] * np.sqrt(lam[keep])

_reference_qcache = {}

def _reference_candidates(fmt, r, b, s, e, window):
    cfg = FORMAT_CONFIGS[fmt]
    block = W64[r, s:e]
    base = baseline_exp(block, fmt)

    exps = []

    for off in sorted(
        range( - window, window + 1),
        key = lambda o: (abs(o), o)
    ):
        exp = base + off

        if "exp_min" in cfg:
            exp = int(
                np.clip(
                    exp,
                    cfg["exp_min"],
                    cfg["exp_max"]
                )
            )

        if exp not in exps:
            exps.append(exp)

    Q = []

    for exp in exps:
        key = (fmt, r, b, exp)

        if key not in _reference_qcache:
            _reference_qcache[key] = quantize_block_at_exp(
                block,
                fmt,
                exp
            )

        Q.append(
            _reference_qcache[key]
        )

    return (
        base,
        np.asarray(exps, dtype = int),
        np.asarray(Q, dtype = np.float64),
    )

def _enumerate_exact(D):
    S = D[0]

    for d in D[1:]:
        S = (
            S[:, None, :]
            + d[None, :, :]
        ).reshape( - 1, S.shape[ - 1])

    sse = np.einsum(
        "ij,ij->i",
        S,
        S
    )

    k = int(
        np.argmin(sse)
    )

    shape = [
        len(d)
        for d in D
    ]

    idx = [
        int(i)
        for i in np.unravel_index(
            k,
            shape
        )
    ]

    return idx, float(sse[k])

def _reference_heuristic(
    D,
    rng,
    n_random,
    pair_moves
):
    """
    Multi-start coordinate descent, optionally followed by pair moves.
    Returns a strong feasible solution; it is not a proof of global optimality.
    """

    nb = len(D)
    K = [
        len(d)
        for d in D
    ]

    sq = [
        np.einsum(
            "ij,ij->i",
            d,
            d
        )
        for d in D
    ]

    starts = [
        [0] * nb,
        [
            int(np.argmin(s))
            for s in sq
        ]
    ]

    starts += [
        [
            int(rng.integers(k))
            for k in K
        ]
        for _ in range(
            n_random
        )
    ]

    best_idx = None
    best = np.inf

    for idx in starts:

        idx = list(idx)

        e = sum(
            D[b][idx[b]]
            for b in range(nb)
        )

        cur = float(
            e @ e
        )

        improved = True

        while improved:
            improved = False

            for b in range(nb):

                base = (
                    e
                    - D[b][idx[b]]
                )

                t = (
                    base[None, :]
                    + D[b]
                )

                s = np.einsum(
                    "ij,ij->i",
                    t,
                    t
                )

                k = int(
                    np.argmin(s)
                )

                if s[k] < cur * (1 - 1e-12):

                    idx[b] = k
                    e = t[k].copy()
                    cur = float(s[k])
                    improved = True

            if (
                not improved
                and pair_moves
            ):

                for b in range(nb):

                    for c in range(
                        b + 1,
                        nb
                    ):

                        base = (
                            e
                            - D[b][idx[b]]
                            - D[c][idx[c]]
                        )

                        t = (
                            base[None, None, :]
                            + D[b][:, None, :]
                            + D[c][None, :, :]
                        )

                        s = np.einsum(
                            "ijk,ijk->ij",
                            t,
                            t
                        )

                        i, j = np.unravel_index(
                            np.argmin(s),
                            s.shape
                        )

                        if s[i, j] < cur * (1 - 1e-12):

                            idx[b] = int(i)
                            idx[c] = int(j)
                            e = t[i, j].copy()
                            cur = float(s[i, j])
                            improved = True

        if cur < best:
            best = cur
            best_idx = list(idx)

    return best_idx, best

def _bfp_tensor_search(X):
    """Exhaustive search of the single tensor-wide Flexpoint exponent
    (window +/-REF_WINDOW around the Autoflex exponent, widened to REF_WINDOW_MAX
    if the optimum sits on the window edge). Objective: actual ||(W - Wq) X||_F^2."""
    t0 = time.perf_counter()
    cfg = FORMAT_CONFIGS["BFP4"]
    base = baseline_exp(W64.ravel(), "BFP4")
    window = REF_WINDOW

    while True:
        exps = sorted(
            {int(np.clip(base + o, cfg["exp_min"], cfg["exp_max"])) for o in range( - window, window + 1)},
            key = lambda e: (abs(e - base), e - base),
        )
        best = None

        for E in exps:
            Wq = quantize_block_at_exp(W64, "BFP4", E)
            R = (W64 - Wq) @ X
            s = float(np.sum(R * R))
            if best is None or s < best[0]:
                best = (s, E, Wq)

        edge = abs(best[1] - base) == window
        if edge and window < REF_WINDOW_MAX:
            window += 1
            continue

        break
    n = W64.shape[0]
    return best[2], {
        "exact_rows": n, "heuristic_rows": 0,
        "widened_rows": n if window > REF_WINDOW else 0,
        "edge_rows": n if edge else 0,
        "offsets": [int(best[1] - base)],
        "seconds": time.perf_counter() - t0,
        "method": "EXACT",
    }

def strong_reference_search(
    fmt,
    X,
    mode = None,
    n_random = None,
    pair_moves = None
):
    """
    mode=None:
        exact when feasible, stronger heuristic otherwise.

    mode='exact':
        force exact enumeration; raises if the memory/search cap is exceeded.

    mode='heuristic':
        force the heuristic search.

    The returned objective is an actual calibration-output SSE objective,
    evaluated efficiently through an exact Gram factor when enabled.

    This is exhaustive over the explicitly bounded exponent candidate window
    (initially +/-3, widened as needed up to +/-5 from the baseline exponent),
    not over every possible E8M0 exponent. The assignment-sized dataset is
    required to use exact enumeration for every row; a heuristic fallback
    raises an error rather than silently passing as brute force.
    """

    n_random = (
        REF_RANDOM_STARTS
        if n_random is None
        else n_random
    )

    pair_moves = (
        REF_PAIR_MOVES
        if pair_moves is None
        else pair_moves
    )

    if fmt == "BFP4" and BFP_SHARING == "tensor":
        return _bfp_tensor_search(X)

    F = (
        _gram_factor(X)
        if REF_USE_GRAM_FACTOR
        else X
    )

    bs = FORMAT_CONFIGS[
        fmt
    ]["block_size"]

    ranges = list(
        rowwise_ranges(
            W64.shape[1],
            bs
        )
    )

    nb = len(ranges)

    feature_dim = max(
        1,
        F.shape[1]
    )

    limit = max(
        1,
        REF_MAX_ELEMS
        // feature_dim
    )

    Wq = np.zeros_like(
        W64
    )

    info = {
        "exact_rows": 0,
        "heuristic_rows": 0,
        "widened_rows": 0,
        "edge_rows": 0,
        "offsets": [],
    }

    t0 = time.perf_counter()

    for r in range(
        W64.shape[0]
    ):

        T = [
            W64[r, s:e]
            @ F[s:e]
            for s, e in ranges
        ]

        rng = np.random.default_rng(
            REF_SEED + r
        )

        window = REF_WINDOW

        while True:

            C = [
                _reference_candidates(
                    fmt,
                    r,
                    b,
                    s,
                    e,
                    window
                )
                for b, (s, e)
                in enumerate(ranges)
            ]

            D = [
                T[b]
                - C[b][2] @ F[s:e]
                for b, (s, e)
                in enumerate(ranges)
            ]

            combinations = math.prod(
                len(d)
                for d in D
            )

            exact = (
                mode == "exact"
                or (
                    mode is None
                    and combinations <= limit
                )
            )

            if (
                exact
                and combinations > limit
            ):
                raise RuntimeError(
                    f"{fmt} row {r}: "
                    f"{combinations:,} combinations "
                    f"exceed exact-search cap {limit:,}"
                )

            if exact:
                idx, _ = _enumerate_exact(
                    D
                )
            else:
                idx, _ = _reference_heuristic(
                    D,
                    rng,
                    n_random,
                    pair_moves
                )

            offsets = [
                int(
                    C[b][1][idx[b]]
                    - C[b][0]
                )
                for b in range(nb)
            ]

            edge = any(
                abs(o) == window
                for o in offsets
            )

            can_widen_exact = (
                (
                    2 * (window + 1)
                    + 1
                ) ** nb
                <= limit
            )

            if (
                edge
                and window < REF_WINDOW_MAX
                and (
                    not exact
                    or can_widen_exact
                )
            ):
                window += 1
                continue

            break

        if exact:
            info[
                "exact_rows"
            ] += 1
        else:
            info[
                "heuristic_rows"
            ] += 1

        info[
            "widened_rows"
        ] += int(
            window > REF_WINDOW
        )

        info[
            "edge_rows"
        ] += int(edge)

        info[
            "offsets"
        ] += offsets

        for b, (s, e) in enumerate(
            ranges
        ):
            Wq[
                r,
                s:e
            ] = C[b][2][
                idx[b]
            ]

    info[
        "seconds"
    ] = (
        time.perf_counter()
        - t0
    )

    if info[
        "heuristic_rows"
    ] == 0:
        info[
            "method"
        ] = "EXACT"
    elif info[
        "exact_rows"
    ] == 0:
        info[
            "method"
        ] = "HEURISTIC (strong feasible upper bound)"
    else:
        info[
            "method"
        ] = "MIXED exact/heuristic"

    return Wq, info


In [8]:
comparison_rows = []

for label, case in CASES.items():
    W = case['W']
    X = case['X']
    W64 = np.asarray(W, dtype = np.float64)
    X64 = np.asarray(X, dtype = np.float64)
    Y64 = W64 @ X64
    output_norm = float(np.linalg.norm(Y64))

    _reference_qcache = {}

    generic = RESULTS[label]
    generic_scores = generic['scores'].copy()
    generic_scores = generic_scores.sort_values('Sampled_SSE', kind = 'stable').reset_index(drop = True)
    generic_scores['Generic_Rank'] = np.arange(1, len(generic_scores) + 1)
    generic_winner = generic['winner']

    baseline_errors = {}
    optimized_errors = {}
    search_methods = {}
    optimized_matrices = {}

    for fmt in REF_FORMATS:
        cfg = FORMATS[fmt]
        base_exp = baseline_exponents(W64, cfg)
        Wq_base = materialize(W64, cfg, base_exp)
        baseline_residual = (W64 - Wq_base) @ X64
        baseline_errors[fmt] = (
            0.0 if output_norm == 0 and np.linalg.norm(baseline_residual) == 0
            else (float('inf') if output_norm == 0
                  else 100.0 * float(np.linalg.norm(baseline_residual)) / output_norm)
        )

        Wq_opt, info = strong_reference_search(fmt, X64)
        if info['heuristic_rows'] != 0:
            raise RuntimeError(
                f"Dataset {label}, format {fmt}: the original strong-reference search "
                f"used a heuristic for {info['heuristic_rows']} row(s). Increase REF_MAX_ELEMS "
                "or reduce the declared candidate window; this comparison requires exact search."
            )

        opt_residual = (W64 - Wq_opt) @ X64
        optimized_errors[fmt] = (
            0.0 if output_norm == 0 and np.linalg.norm(opt_residual) == 0
            else (float('inf') if output_norm == 0
                  else 100.0 * float(np.linalg.norm(opt_residual)) / output_norm)
        )
        search_methods[fmt] = info['method']
        optimized_matrices[fmt] = Wq_opt

        _reference_qcache = {}

    baseline_winner = min(REF_FORMATS, key = lambda f: baseline_errors[f])
    brute_winner = min(REF_FORMATS, key = lambda f: optimized_errors[f])
    generic_rank = dict(zip(generic_scores['Format'], generic_scores['Generic_Rank']))
    generic_err = dict(zip(generic_scores['Format'], generic_scores['Sampled_error_pct']))

    for fmt in REF_FORMATS:
        comparison_rows.append({
            'Dataset': label,
            'Format': fmt,
            'Generic_Sampled_Rank': int(generic_rank[fmt]),
            'Generic_Sampled_Error_%': float(generic_err[fmt]),
            'Generic_Predicted_Winner': fmt == generic_winner,
            'Full_Baseline_Error_%': float(baseline_errors[fmt]),
            'BruteForce_Optimized_Error_%': float(optimized_errors[fmt]),
            'BruteForce_Optimized_Rank': int(1 + sum(optimized_errors[g] < optimized_errors[fmt] for g in REF_FORMATS)),
            'BruteForce_Search': search_methods[fmt],
            'BruteForce_Winner': fmt == brute_winner,
            'Generic_Winner_Matches_BruteForce': generic_winner == brute_winner,
        })

COMPARISON_TABLE = pd.DataFrame(comparison_rows)
COMPARISON_TABLE = COMPARISON_TABLE.sort_values(['Dataset', 'BruteForce_Optimized_Rank', 'Format'], kind = 'stable').reset_index(drop = True)

OUTPUT_DIR.mkdir(parents = True, exist_ok = True)
COMPARISON_TABLE.to_csv(OUTPUT_DIR / 'generic_vs_bruteforce_comparison.csv', index = False)

display(COMPARISON_TABLE.round(6))


,Dataset,Format,Generic_Sampled_Rank,Generic_Sampled_Error_%,Generic_Predicted_Winner,Full_Baseline_Error_%,BruteForce_Optimized_Error_%,BruteForce_Optimized_Rank,BruteForce_Search,BruteForce_Winner,Generic_Winner_Matches_BruteForce
0,Dataset C,E2M1,1,12.551902,True,12.552314,12.030660,1,EXACT,True,True
1,Dataset C,BFP4,3,14.613751,False,15.807957,13.452060,2,EXACT,False,True
2,Dataset C,MXINT4,2,14.613751,False,15.807957,13.452060,2,EXACT,False,True
3,Dataset C,MXUINT4,5,15.743917,False,17.126238,13.452060,2,EXACT,False,True
4,Dataset C,E1M2,4,14.615353,False,15.817232,14.004212,5,EXACT,False,True
